# HW10 Key

In [ ]:
##### 5 pts #####
import numpy as np
import seaborn as sns
from sklearn.datasets import make_gaussian_quantiles
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

## 1. SGD for classification (50 pts)
Write `NN_clf_SGD()` such that 
- SGD is added in NN_clf() that was in your HW9. (should be able to use the `simple_NN_SGD()` function in [NN2](https://github.com/xuemeic/MTDS/blob/main/notebooks/NN_from_scratch2.ipynb))
- You should have options for choosing different gradient descent schemes: 'regular', 'momentum', 'AdaGrad','RMSprop'

In [2]:
def relu(x):
    # relu function
    return np.maximum(x, 0)
def sigmoid(x):
    return 1 / (1 + np.exp(-x))
def tanh(x):
    return (np.exp(x) - np.exp(-x))/(np.exp(x) + np.exp(-x))
def ce_J(s, y):
    '''Cross Entropy Function
    '''
    # both s, y are made column vectors
    s = s.reshape([-1, 1])
    y = s.reshape([-1, 1])
    N = len(s)
    # clipping is for stability
    epsilon = 1e-15
    s_clipped = np.clip(s, epsilon, 1 - epsilon)
    # s_clipped willl stay at least epsilon away from 0 or 1
    return - (np.sum(y*np.log(s_clipped) + (1-y)*np.log(1-s_clipped)))/N

In [3]:
def forward_prop(X, theta, activate_func):
    W1, W2, b, b2 = theta
    u1 = X@W1 + b
    h = activate_func(u1) # relu or tanh
    u2 = h@W2 + b2
    s = sigmoid(u2)
    return u1, h, u2, s

In [4]:
def NN_clf_pred(X, theta_trained, activate_func):
    # X is new data
    _, _, _, s = forward_prop(X, theta_trained, activate_func)
    return s
def NN_clf_pred_class(X, theta_trained, activate_func):
    s = NN_clf_pred(X, theta_trained, activate_func)
    return (s > 0.5).astype(int)

In [5]:
def NN_clf_SGD(X, y, hidden_size = 5, seed = 3, bs = 100, lr = 1e-1, n_epoch = 30, activation_func = relu, **kwargs):
    '''
    X: features. numpy array. each row is an observation.
    y: binary numpy array.
    W1_init: optional argument for user-initialized weight
    W2_init: optional argument for user-initialized weight
    b_init: optional argument for user-initialized weight
    b2_init: optional argument for user-initialized weight
    gd: optional argument for gradient descent method. Choices are 
        'regular'
        'momentum': optional arg "mu", default is 0.95
        'AdaGrad':
        'RMSprop':
    '''
    n, p = X.shape

    # size (number of nodes) of hidden layer
    p1 = hidden_size
    
    # turn y into a column
    y = y.reshape([-1, 1])
    
    gd = kwargs.get('gd', 'regular')
    # initialization and hyperparameter for advanced gradient descent methods
    if gd == 'momentum':
        v1 = 0
        v2 = 0
        vb = 0
        vb2 = 0
        mu = kwargs.get('mu', 0.95)
    elif gd == 'AdaGrad':
        r1 = 0
        r2 = 0
        rb = 0
        rb2 = 0
    elif gd == 'RMSprop':
        rho = kwargs.get('rho', 0.9)
        r1 = 0
        r2 = 0
        rb = 0
        rb2 = 0

    # random initialization if initial weights not provided
    np.random.seed(seed)
    w_scale = np.sqrt(6/(p+1))
    W1_rand = np.random.rand(p, p1) * w_scale 
    W2_rand = np.random.rand(p1, 1) * w_scale
    # bias terms are usually initialized as 0
    b = 0
    b2 = 0

    W1 = kwargs.get('W1_init', W1_rand)
    W2 = kwargs.get('W2_init', W2_rand)
    b = kwargs.get('b_init', 0)
    b2 = kwargs.get('b2_init', 0)

    # number of batches
    n_bt = int(np.ceil(n/bs))
    J_all = np.zeros(n_epoch)
    for ei in range(n_epoch):
        # ei for epoch index
        # for each epoch
        # first shuffle X
        shuffled_idx = np.random.permutation(n)

        # the following for loop runs through the whole training set ONCE
        for bi in range(n_bt):
            # for the last batch, (bi+1)*bs most likely exceeds n. this is ok as numpy will automatically take the min of {n, (bi+1)*bs}
            bt_idx = shuffled_idx[(bi*bs):(bi+1)*bs] 
            X_bt = X[bt_idx,:]
            y_bt = y[bt_idx]
            # forward propargation
            u1, h, u2, s = forward_prop(X_bt, (W1, W2, b, b2), activation_func)
            # compute cost
            J = ce_J(s, y_bt)
            
            # backward propagation for computing gradient 
            du2 = -1/n*(y_bt - s)
            dW2 = (h.T)@du2
        
            dh = du2@W2.T
            if activation_func == relu:
                du1 = dh * relu(np.sign(u1))
            elif activation_func == tanh:
                du1 = dh * (1-h**2)
            dW1 = X_bt.T@du1
        
            db = np.sum(du1, axis = 0)
            db2 = np.sum(du2)

            # gradient descent 
            if gd == 'momentum':
                # if mu = 0, then this is the same as simple_NN() 
                v1 = mu*v1 - lr*dW1
                v2 = mu*v2 - lr*dW2
                vb = mu*vb - lr*db
                vb2 = mu*vb2 - lr*db2
                W1 = W1 + v1
                W2 = W2 + v2
                b = b + vb
                b2 = b2 + vb2
            elif gd == 'regular':
                W1 = W1 - lr * dW1
                W2 = W2 - lr * dW2
                b = b - lr * db
                b2 = b2 - lr * db2
            elif gd == 'AdaGrad':
                r1 = r1 + dW1**2
                r2 = r2 + dW2**2
                rb = rb + db**2
                rb2 = rb2 + db2**2
                W1 = W1 - lr/(1e-7 + r1**0.5)*dW1
                W2 = W2 - lr/(1e-7 + r2**0.5)*dW2
                b = b - lr/(1e-7 + rb**0.5)*db
                b2 = b2 - lr/(1e-7 + rb2**0.5)*db2
            elif gd == 'RMSprop':
                r1 = rho*r1 + (1-rho) * dW1**2
                r2 = rho*r2 + (1-rho) * dW2**2
                rb = rho*rb + (1-rho) * db**2
                rb2 = rho*rb2 + (1-rho) * db2**2
                W1 = W1 - lr/(1e-7 + r1**0.5)*dW1
                W2 = W2 - lr/(1e-7 + r2**0.5)*dW2
                b = b - lr/(1e-7 + rb**0.5)*db
                b2 = b2 - lr/(1e-7 + rb2**0.5)*db2
        # collecting cost after an epoch
        theta = (W1, W2, b, b2)
        _,_,_, s_all = forward_prop(X, theta, activation_func)
        J_all[ei] = ce_J(s_all, y)
    theta = (W1, W2, b, b2)
    para = (gd, bs, n_epoch)
    return theta, J_all, para

## 2. (25 pts) Test this function above using the same simulated data from HW9. Try different batch size and different learning rate schemes.

In [6]:
N = 500
X, y = make_gaussian_quantiles(mean = None, cov = 0.5, n_samples = N, 
                               n_features = 2, n_classes = 2, shuffle = True, 
                               random_state = 42)

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 45)

In [8]:
theta_trained, J, para = NN_clf_SGD(X_train, y_train, lr = 1e0, bs = X_train.shape[0], n_epoch=20)
y_pred_01 = NN_clf_pred_class(X_test, theta_trained, relu)
print(f"gd is {para[0]}; batch_size = {para[1]}; n_epochs = {para[2]}; accuracy, {accuracy_score(y_test, y_pred_01)}")

gd is regular; batch_size = 400; n_epochs = 20; accuracy, 0.69


In [10]:
J

array([0.61362345, 0.62591209, 0.63191482, 0.63391521, 0.63480021,
       0.63536725, 0.63502461, 0.63438848, 0.63415677, 0.63360149,
       0.63276895, 0.63182628, 0.63086955, 0.63007779, 0.6288909 ,
       0.62777602, 0.62706556, 0.6257178 , 0.62439786, 0.62339741])

In [11]:
theta_trained, J, para = NN_clf_SGD(X_train, y_train, lr = 1e0, bs = 20, n_epoch=20)
y_pred_01 = NN_clf_pred_class(X_test, theta_trained, relu)
print(f"gd is {para[0]}; batch_size = {para[1]}; n_epochs = {para[2]}; accuracy, {accuracy_score(y_test, y_pred_01)}")

gd is regular; batch_size = 20; n_epochs = 20; accuracy, 0.69


In [12]:
theta_trained, J, para = NN_clf_SGD(X_train, y_train, lr = 1e0, bs = 20, n_epoch=20, gd = 'momentum')
y_pred_01 = NN_clf_pred_class(X_test, theta_trained, relu)
print(f"gd is {para[0]}; batch_size = {para[1]}; n_epochs = {para[2]}; accuracy, {accuracy_score(y_test, y_pred_01)}")

gd is momentum; batch_size = 20; n_epochs = 20; accuracy, 0.98


In [13]:
theta_trained, J, para = NN_clf_SGD(X_train, y_train, lr = 1e0, bs = 20, n_epoch = 20, gd = 'AdaGrad')
y_pred_01 = NN_clf_pred_class(X_test, theta_trained, relu)
print(f"gd is {para[0]}; batch_size = {para[1]}; n_epochs = {para[2]}; accuracy, {accuracy_score(y_test, y_pred_01)}")

gd is AdaGrad; batch_size = 20; n_epochs = 20; accuracy, 0.98


In [14]:
theta_trained, J, para = NN_clf_SGD(X_train, y_train, lr = 1e0, bs = 20, n_epoch = 4, gd = 'RMSprop')
y_pred_01 = NN_clf_pred_class(X_test, theta_trained, relu)
print(f"gd is {para[0]}; batch_size = {para[1]}; n_epochs = {para[2]}; accuracy, {accuracy_score(y_test, y_pred_01)}")

gd is RMSprop; batch_size = 20; n_epochs = 4; accuracy, 0.74


## 3. (20 pts) Repeat 1 and 2 where you replace relu with the hyperbolic tangent activation function.

$$tanh(z)=\frac{e^{z}-e^{-z}}{e^{z}+e^{-z}}$$

In [15]:
theta_trained, J, para = NN_clf_SGD(X_train, y_train, lr = 1e0, bs = X_train.shape[0], n_epoch=20, activation_func=tanh)
y_pred_01 = NN_clf_pred_class(X_test, theta_trained, tanh)
print(f"gd is {para[0]}; batch_size = {para[1]}; n_epochs = {para[2]}; accuracy, {accuracy_score(y_test, y_pred_01)}")

gd is regular; batch_size = 400; n_epochs = 20; accuracy, 0.77


In [16]:
J

array([0.59041079, 0.62907583, 0.6533069 , 0.66767054, 0.67598836,
       0.68081056, 0.68364057, 0.68531504, 0.68629238, 0.68682599,
       0.68705883, 0.68707335, 0.68691803, 0.68662175, 0.68620189,
       0.68566908, 0.68502991, 0.6842887 , 0.68344846, 0.68251151])

In [17]:
theta_trained, J, para = NN_clf_SGD(X_train, y_train, lr = 1e0, bs = 20, n_epoch=20, activation_func=tanh)
y_pred_01 = NN_clf_pred_class(X_test, theta_trained, tanh)
print(f"gd is {para[0]}; batch_size = {para[1]}; n_epochs = {para[2]}; accuracy, {accuracy_score(y_test, y_pred_01)}")

gd is regular; batch_size = 20; n_epochs = 20; accuracy, 0.76


In [18]:
theta_trained, J, para = NN_clf_SGD(X_train, y_train, lr = 1e0, bs = 20, n_epoch=20, gd = 'momentum', activation_func=tanh)
y_pred_01 = NN_clf_pred_class(X_test, theta_trained, tanh)
print(f"gd is {para[0]}; batch_size = {para[1]}; n_epochs = {para[2]}; accuracy, {accuracy_score(y_test, y_pred_01)}")

gd is momentum; batch_size = 20; n_epochs = 20; accuracy, 0.96


In [19]:
theta_trained, J, para = NN_clf_SGD(X_train, y_train, lr = 1e0, bs = 20, n_epoch = 20, gd = 'AdaGrad', activation_func=tanh)
y_pred_01 = NN_clf_pred_class(X_test, theta_trained, tanh)
print(f"gd is {para[0]}; batch_size = {para[1]}; n_epochs = {para[2]}; accuracy, {accuracy_score(y_test, y_pred_01)}")

gd is AdaGrad; batch_size = 20; n_epochs = 20; accuracy, 0.97


In [20]:
theta_trained, J, para = NN_clf_SGD(X_train, y_train, lr = 1e0, bs = 20, n_epoch = 4, gd = 'RMSprop', activation_func=tanh)
y_pred_01 = NN_clf_pred_class(X_test, theta_trained, tanh)
print(f"gd is {para[0]}; batch_size = {para[1]}; n_epochs = {para[2]}; accuracy, {accuracy_score(y_test, y_pred_01)}")

gd is RMSprop; batch_size = 20; n_epochs = 4; accuracy, 0.91
